This code essentially loops through all the tickers and applies the individual indicator code to all of them, 
then that data is appended to their respective dataframes.

### Integrate covariance and sector code !!! Don't forget leon or whoever


### Key assumptions: 
1.) Tickers passed through have passed the initial filter

2.) Price data has been downloaded and put in its respective dataframe pertaining specifically to the ticker.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

In [ ]:
for ticker, ticker_data in stock_data.items():
    stock_data[ticker] = indicator_application(ticker, ticker_data, start_date)
    


In [18]:
def indicator_application(ticker, ticker_df, start_date):
    ticker = yf.Ticker(ticker)

    ticker_df["Market Cap"] = pull_market_cap(ticker, start_date)

    ticker_df["BiWeekly Price STD"] = avg_price_std(ticker_df)

    ticker_df["Beta"] = pull_beta(ticker)

    ticker_df = pd.concat([ticker_df, calculate_adx(ticker_df)], axis=1)

    return ticker_df

# Should return a df[H,L,C,V,Marketcap,BIWPSTD,BETA,adx, DX, +DI, -DI,smoothed atr




In [19]:
def pull_market_cap(ticker_symbol, target_date):

  target_dt = pd.to_datetime(target_date)
  end_dt = target_dt + pd.Timedelta(days=5)

  hist = ticker.history(start=target_dt.strftime("%Y-%m-%d"), end=end_dt.strftime("%Y-%m-%d"))

  close_price = hist.iloc[0]["Close"]

  shares_history = ticker.get_shares_full(start=target_dt.strftime("%Y-%m-%d"), end=end_dt.strftime("%Y-%m-%d"))

  if shares_history is not None:
      share_count = shares_history.iloc[0]
  else:
      share_count = ticker.info.get("sharesOutstanding") #not sure what else to do but just put my hands together and pray no share split


  market_cap = close_price * share_count
  return market_cap


In [15]:
def calculate_adx(data, period=14):

    data = data[["High", "Low", "Close"]].copy()

    # 1. True Range
    high_low = data["High"] - data["Low"]
    high_close = abs(data["High"] - data["Close"].shift(1))
    low_close = abs(data["Low"] - data["Close"].shift(1))

    data["Trading Range"] = high_low
    data["up gap"] = high_close
    data["down gap"] = low_close

    data["True Range"] = pd.concat(
        [high_low, high_close, low_close], axis=1
    ).max(axis=1)

    # 2. Directional Momentum
    up_move = data["High"].diff()
    down_move = -data["Low"].diff()

    pos_cond = (up_move > down_move) & (up_move > 0)
    neg_cond = (down_move > up_move) & (down_move > 0)

    data["+DM"] = np.where(pos_cond, up_move, 0)
    data["-DM"] = np.where(neg_cond, down_move, 0)

    # 3. Wilder Smoothing
    def wilder_smoothing(series, smoothing_period):
        smoothed = {}
        initial = sum(series.values[1:smoothing_period + 1])
        smoothed[smoothing_period] = initial

        for i in range(smoothing_period + 1, len(series)):
            smoothed[i] = (
                smoothed[i - 1]
                - (smoothed[i - 1] / smoothing_period)
                + series.values[i])

        return pd.Series(list(smoothed.values()), index=series.index[smoothing_period:])

    data["smoothed +DM"] = wilder_smoothing(data["+DM"], period)
    data["smoothed -DM"] = wilder_smoothing(data["-DM"], period)
    data["smoothed TR"] = wilder_smoothing(data["True Range"], period)

    # 4. Directional Index
    data["+DI"] = (data["smoothed +DM"] / data["smoothed TR"]) * 100
    data["-DI"] = (data["smoothed -DM"] / data["smoothed TR"]) * 100

    data["DX"] = (100 * abs(data["+DI"] - data["-DI"])) / (abs(data["+DI"]) + abs(data["-DI"]))

    # 5. Average Directional Index
    adx_values = {}
    initial = data["DX"].iloc[period:2 * period].mean()

    adx_values[2 * period - 1] = initial

    for i in range(2 * period, len(data)):
        adx_values[i] = ((period - 1) * adx_values[i - 1] + data["DX"].iloc[i]) / period

    data["adx"] = pd.Series(list(adx_values.values()), index=data.index[2 * period - 1:])

    data = data[["adx", "DX", "+DI", "-DI", "smoothed TR"]]

    return data


In [16]:
def avg_price_std(data):

    biweekly_std = data["Close"].resample("2W").std()

    avg_std = biweekly_std.dropna().mean()

    return avg_std




In [20]:
def pull_beta(ticker):

    info = ticker.info
    beta = info.get("beta")

    return beta